In [ ]:
!pip uninstall datasets
!pip install datasets==3.6

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Would remove:
    /usr/local/bin/datasets-cli
    /usr/local/lib/python3.12/dist-packages/datasets-4.0.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/datasets/*
Proceed (Y/n)? Y
  Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.5 MB/s eta 0:00:00


In [ ]:
import os

base_dir = "/content/ecommerce-agent"
folders = ["data", "notebooks", "src", "vectorstore", "outputs", "config"]

for folder in folders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

# Archivos base en src
files = ["__init__.py", "data_loader.py", "embeddings_builder.py", "rag_pipeline.py"]
for f in files:
    open(os.path.join(base_dir, "src", f), "a").close()

print("✅ Estructura de proyecto creada en:", base_dir)


✅ Estructura de proyecto creada en: /content/ecommerce-agent


In [ ]:
from datasets import load_dataset
import pandas as pd
import os

# Escoge la categoría que quieras ("All_Beauty" es solo un ejemplo)
review_config = "raw_review_All_Beauty"
meta_config   = "raw_meta_All_Beauty"

# Directorios
base_dir = "/content/ecommerce-agent/data"
os.makedirs(base_dir, exist_ok=True)

# 1️⃣ Cargar reseñas
dataset_reviews = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    review_config,
    split="full", # Changed from "train" to "full"
    trust_remote_code=True
)

df_reviews = pd.DataFrame(dataset_reviews)

print("Reseñas columnas:", df_reviews.columns.tolist())
print("Ejemplo reseña:", df_reviews.head(2))

# Guardar reseñas
reviews_json = os.path.join(base_dir, f"{review_config}.json")
df_reviews.to_json(reviews_json, orient="records", lines=True)
print("Reseñas guardadas en:", reviews_json)

# 2️⃣ Cargar metadatos
dataset_meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    meta_config,
    split="full",
    trust_remote_code=True
)

df_meta = pd.DataFrame(dataset_meta)

print("Metadatos columnas:", df_meta.columns.tolist())
print("Ejemplo metadato:", df_meta.head(2))

# Guardar metadatos
meta_json = os.path.join(base_dir, f"{meta_config}.json")
df_meta.to_json(meta_json, orient="records", lines=True)
print("Metadatos guardados en:", meta_json)

In [ ]:
import os

# Ruta base del proyecto
base_dir = "/content/ecommerce-agent"

# Subcarpetas principales
folders = [
    "data/raw",            # datasets descargados (sin limpiar)
    "data/processed",      # datos limpios y listos para embeddings
    "src",                 # scripts principales (data_loader, embeddings, rag, etc.)
    "vectorstore/chroma_db",  # almacenamiento de embeddings
    "outputs/logs",        # logs y resultados
    "outputs/reports",     # informes o métricas
    "notebooks",           # experimentos Jupyter / Colab
    "config",              # configuración de API Keys o parámetros
]

# Crear todas las carpetas
for folder in folders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

print("✅ Estructura de carpetas creada correctamente en:", base_dir)

# Mostrar estructura resultante
for root, dirs, files in os.walk(base_dir):
    level = root.replace(base_dir, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")


✅ Estructura de carpetas creada correctamente en: /content/ecommerce-agent
ecommerce-agent/
    notebooks/
    config/
    vectorstore/
        chroma_db/
    data/
        raw_meta_All_Beauty.json
        raw_review_All_Beauty.json
        processed/
        raw/
    outputs/
        logs/
        reports/
    src/
        __init__.py
        data_loader.py
        embeddings_builder.py
        rag_pipeline.py


In [ ]:
import pandas as pd
import os
import json

# f4c2 Rutas
# Los archivos fueron guardados directamente en /content/ecommerce-agent/data por la celda M6_U8TgLQHEP
data_storage_dir = "/content/ecommerce-agent/data"
processed_dir = "/content/ecommerce-agent/data/processed"
os.makedirs(processed_dir, exist_ok=True)

# Actualizar rutas para que apunten a los archivos correctos
meta_path = os.path.join(data_storage_dir, "raw_meta_All_Beauty.json")
review_path = os.path.join(data_storage_dir, "raw_review_All_Beauty.json")

# --- Cargar datasets ---
print(" Cargando metadatos y reseñas...")
meta = pd.read_json(meta_path, lines=True)
reviews = pd.read_json(review_path, lines=True)

print(f"Metadatos: {len(meta)} registros")
print(f"Reseñas: {len(reviews)} registros")

# --- Extraer detalles del producto (brand, color, size) ---
def extract_detail(detail_dict, key):
    if isinstance(detail_dict, dict):
        return detail_dict.get(key)
    elif isinstance(detail_dict, list) and len(detail_dict) > 0 and isinstance(detail_dict[0], dict):
        return detail_dict[0].get(key)
    return None

meta['brand'] = meta['details'].apply(lambda x: extract_detail(x, 'Brand'))
meta['color'] = meta['details'].apply(lambda x: extract_detail(x, 'Color'))
meta['size'] = meta['details'].apply(lambda x: extract_detail(x, 'Size')) # Corrected line to extract 'Size'


 Cargando metadatos y reseñas...
Metadatos: 112590 registros
Reseñas: 701528 registros


In [ ]:
import pandas as pd
import os

# Rutas de archivos crudos
# Los archivos fueron guardados directamente en /content/ecommerce-agent/data por la celda M6_U8TgLQHEP
data_storage_dir = "/content/ecommerce-agent/data"
processed_dir = "/content/ecommerce-agent/data/processed"
os.makedirs(processed_dir, exist_ok=True)

meta_path = os.path.join(data_storage_dir, "raw_meta_All_Beauty.json")
review_path = os.path.join(data_storage_dir, "raw_review_All_Beauty.json")

# Cargar datasets
meta = pd.read_json(meta_path, lines=True)
reviews = pd.read_json(review_path, lines=True)

# Función para extraer Brand, Color y Size desde 'details'
def extract_detail(detail_dict, key):
    if isinstance(detail_dict, dict):
        return detail_dict.get(key)
    elif isinstance(detail_dict, list) and len(detail_dict) > 0 and isinstance(detail_dict[0], dict):
        return detail_dict[0].get(key)
    return None

meta['brand'] = meta['details'].apply(lambda x: extract_detail(x, 'Brand'))
meta['color'] = meta['details'].apply(lambda x: extract_detail(x, 'Color'))
meta['size']  = meta['details'].apply(lambda x: extract_detail(x, 'Size'))

# Seleccionar columnas importantes
meta_clean = meta[['parent_asin', 'title', 'description', 'price', 'brand', 'color', 'size']]
reviews_clean = reviews[['parent_asin', 'rating', 'text']]

# Combinar metadatos y reseñas
merged = pd.merge(reviews_clean, meta_clean, on='parent_asin', how='left')

# Calcular promedio de rating por producto
avg_rating = merged.groupby('parent_asin')['rating'].mean().reset_index().rename(columns={'rating': 'avg_rating'})
meta_final = pd.merge(meta_clean, avg_rating, on='parent_asin', how='left')

# Filtrar solo productos con reseñas
meta_final = meta_final.dropna(subset=['avg_rating'])

# Seleccionar columnas finales
final_columns = ['title', 'description', 'price', 'brand', 'color', 'size', 'avg_rating']
df_final = meta_final[final_columns].fillna('')

# Guardar dataset limpio
output_path = os.path.join(processed_dir, "amazon_products_with_reviews.json")
df_final.to_json(output_path, orient='records', indent=2, force_ascii=False)

print("Dataset combinado creado correctamente en:", output_path)
print("Total productos:", len(df_final))

Dataset combinado creado correctamente en: /content/ecommerce-agent/data/processed/amazon_products_with_reviews.json
Total productos: 112565


In [ ]:
import pandas as pd
import os

# Ruta del dataset limpio
data_path = "/content/ecommerce-agent/data/processed/amazon_products_with_reviews.json"

# Cargar dataset
df = pd.read_json(data_path, lines=False)

# Mostrar todas las columnas
print("Columnas disponibles:")
for i, col in enumerate(df.columns):
    print(f"{i+1}. {col}")

# Vista previa de los primeros registros
print("\nPrimeros registros del dataset:")
print(df.head(5))

# Información general sobre columnas y tipos de datos
print("\nInformación general del dataframe:")
print(df.info())

# Estadísticas básicas
print("\nEstadísticas básicas:")
print(df.describe(include='all'))


Columnas disponibles:
1. title
2. description
3. price
4. brand
5. color
6. size
7. avg_rating

Primeros registros del dataset:
                                               title  \
0  Howard LC0008 Leather Conditioner, 8-Ounce (4-...   
1  Yes to Tomatoes Detoxifying Charcoal Cleanser ...   
2   Eye Patch Black Adult with Tie Band (6 Per Pack)   
3  Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...   
4  Precision Plunger Bars for Cartridge Grips – 9...   

                                         description price brand color size  \
0                                                 []  None                    
1                                                 []  None                    
2                                                 []  None                    
3                                                 []  None                    
4  [The Precision Plunger Bars are designed to wo...  None                    

   avg_rating  
0    4.600000  
1    5.000000  
2    4.35714

In [ ]:
df['description'] = df['description'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)


In [ ]:
import pandas as pd
import os

processed_dir = "/content/ecommerce-agent/data"
meta_path = os.path.join(processed_dir, "raw_meta_All_Beauty.json")

meta = pd.read_json(meta_path, lines=True)

print("Columnas disponibles en metadatos:")
print(meta.columns.tolist())

# Revisar un registro para inspeccionar la estructura real
print("\nEjemplo de registro:")
print(meta.iloc[0])


Columnas disponibles en metadatos:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

Ejemplo de registro:
main_category                                             All Beauty
title              Howard LC0008 Leather Conditioner, 8-Ounce (4-...
average_rating                                                   4.8
rating_number                                                     10
features                                                          []
description                                                       []
price                                                           None
images             {'hi_res': [None, 'https://m.media-amazon.com/...
videos                       {'title': [], 'url': [], 'user_id': []}
store                                                Howard Products
categories                                   

In [ ]:
import pandas as pd
import os
import json

# Directorios
data_dir = "/content/mydrive/MyDrive/ecommerce-agent/data"
processed_dir = os.path.join(data_dir, "processed")
os.makedirs(processed_dir, exist_ok=True)

# Archivos originales
meta_path = os.path.join(data_dir, "raw_meta_All_Beauty.json")
review_path = os.path.join(data_dir, "raw_review_All_Beauty.json")

# 1️⃣ Cargar metadatos y reseñas
meta = pd.read_json(meta_path, lines=True)
reviews = pd.read_json(review_path, lines=True)

# 2️⃣ Extraer Brand, Color, Size desde 'details' (JSON string)
def extract_detail(detail_str, key):
    if isinstance(detail_str, str):
        try:
            detail_dict = json.loads(detail_str.replace("'", '"'))
            return detail_dict.get(key, '')
        except:
            return ''
    elif isinstance(detail_str, dict):
        return detail_str.get(key, '')
    return ''

meta['brand'] = meta['details'].apply(lambda x: extract_detail(x, 'Brand'))
meta['color'] = meta['details'].apply(lambda x: extract_detail(x, 'Color'))
meta['size']  = meta['details'].apply(lambda x: extract_detail(x, 'Size'))

# 3️⃣ Limpiar descripción y features
meta['description'] = meta['description'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)
meta['features'] = meta['features'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

# 4️⃣ Crear campo 'text' para embeddings
meta['text'] = meta.apply(lambda row: ' '.join([
    str(row.get('title', '')),
    str(row.get('description', '')),
    str(row.get('features', '')),
    str(row.get('brand', '')),
    str(row.get('color', '')),
    str(row.get('size', ''))
]), axis=1)

# 5️⃣ Calcular avg_rating
# Usamos average_rating de metadata si existe
meta['avg_rating'] = meta['average_rating'].fillna(0)

# 6️⃣ Seleccionar columnas finales
final_columns = ['title', 'description', 'size', 'color', 'text', 'price', 'brand', 'avg_rating'] # Removed duplicate 'color' and 'size'
df_final = meta[final_columns].fillna('')

# 7️⃣ Guardar dataset limpio
output_path = os.path.join(processed_dir, "amazon_products_clean.json")
df_final.to_json(output_path, orient='records', lines=True, force_ascii=False)

print("Dataset limpio creado en:", output_path)
print("Total productos:", len(df_final))

# 8️⃣ Revisar algunas filas
print("\nPrimeros registros:")
print(df_final.head(5))


Dataset limpio creado en: /content/mydrive/MyDrive/ecommerce-agent/data/processed/amazon_products_clean.json
Total productos: 112590

Primeros registros:
                                               title  \
0  Howard LC0008 Leather Conditioner, 8-Ounce (4-...   
1  Yes to Tomatoes Detoxifying Charcoal Cleanser ...   
2   Eye Patch Black Adult with Tie Band (6 Per Pack)   
3  Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...   
4  Precision Plunger Bars for Cartridge Grips – 9...   

                                         description size color  \
0                                                                 
1                                                                 
2                                                                 
3                                                                 
4  The Precision Plunger Bars are designed to wor...              

                                                text price     brand  \
0  Howard LC0008 Leather Condition

In [ ]:
# Función para verificar si el precio es un número válido
def is_valid_price(x):
    try:
        return float(x) > 0
    except:
        return False

# Filtrar productos con precio válido y avg_rating >= 4.0
df_filtered = df_final[df_final['price'].apply(is_valid_price) & (df_final['avg_rating'] >= 4.0)].reset_index(drop=True)

print("Total productos con precio válido y top rating:", len(df_filtered))
print(df_filtered[['title', 'price', 'avg_rating']].head(5))


Total productos con precio válido y top rating: 12837
                                               title  price  avg_rating
0  BioMiracle StarDust Pixie Bubble Mask, Clarify...   5.99         4.4
1  Garnier Fructis Color Sealer, Instant, Lightwe...   24.0         4.4
2  Enjoy VOLUMIZING ELIXIR, Style (with Sleek Ste...  22.49         5.0
3  Suavecito X Breast Cancer Solutions - Original...  11.99         4.8
4  Sharonelle Natural Cream Soft Wax for Sensitiv...   50.0         4.0


In [ ]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key="OPENAI_API_KEY"
)



In [ ]:
import pandas as pd
import os

# Ruta al dataset filtrado previamente (productos con precio y top rating)
processed_path = "/content/ecommerce-agent/data/processed/amazon_products_clean.json"

# Cargar dataset
df_final = pd.read_json(processed_path, lines=True)

# Filtrar solo productos que tengan precio válido y buen rating
df_filtered = df_final[(df_final['price'] != '') & (df_final['price'].notna()) & (df_final['avg_rating'] >= 4.5)].reset_index(drop=True)
print("Total productos con precio y top rating:", len(df_filtered))

# Revisar primeros productos
print(df_filtered[['title', 'price', 'avg_rating']].head(5))



Total productos con precio y top rating: 31004
                                               title  price  avg_rating
0  Howard LC0008 Leather Conditioner, 8-Ounce (4-...   None         4.8
1  Yes to Tomatoes Detoxifying Charcoal Cleanser ...   None         4.5
2  Zydeco Chop Chop Cajun Seasoning Base, 8 Ounce...   None         4.7
3  Enjoy VOLUMIZING ELIXIR, Style (with Sleek Ste...  22.49         5.0
4  Suavecito X Breast Cancer Solutions - Original...  11.99         4.8


In [ ]:
from google.colab import drive
drive.mount('/content/mydrive/')

# Carpeta donde están tus datos

data_path = '/content/mydrive/MyDrive/ecommerce-agent'

Mounted at /content/mydrive/


In [ ]:
#read the json amazon_products_with_reviews.kspm ==>t

In [ ]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')

In [ ]:
# In terminal/Jupyter cell (replace with your key)
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
%%bash
pip install langchain langchain-community langchain-chroma langchain-openai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.


In [ ]:
import json
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
import chromadb

# 🔹 Definir función para formatear los documentos
def format_docs(docs):
    formatted = []
    for doc in docs:
        meta = doc.metadata
        precio = meta.get("price", "Sin precio")
        rating = meta.get("rating", "Sin rating")
        title = meta.get("title", "Sin título")
        descripcion = meta.get("description", "Sin descripción")
        size = meta.get("size", "Sin tamaño")
        color = meta.get("color", "Sin color")
        formatted.append(f"{title} | Precio: {precio} | Rating: {rating}")
    return "\n\n".join(formatted)

# 🔹 Configurar conexión con Chroma Cloud
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = chromadb.CloudClient(
    CHROMA_API_KEY= userdata.get('CHROMA_API_KEY'),
    TENANT_KEY= userdata.get('TENANT_KEY'),
    database='Amazon_database'
)

vectorstore = Chroma(
    client=client,
    collection_name="langchain",
    embedding_function=embeddings
)

# 🔹 Leer el archivo JSON procesado
json_path = "/content/mydrive/MyDrive/ecommerce-agent/data/processed/amazon_products_clean.json"

with open(json_path, "r", encoding="utf-8") as f:
    # Modified: Read JSONL file line by line
    data = [json.loads(line) for line in f]

# 🔹 Convertir cada ítem en un Document con metadatos
docs = []
for item in data:
    # Texto base que se embeddeará (título + descripción)
    content = f"{item.get('title', '')}\n\n{item.get('description', '')}"

    # Truncate description for metadata to avoid quota issues
    truncated_description = item.get("description", "")
    if len(truncated_description) > 500:
        truncated_description = truncated_description[:500] + "..."

    # Crear objeto Document con contenido y metadatos
    doc = Document(
        page_content=content,
        metadata={
            "price": item.get("price"),
            "avg_rating": item.get("avg_rating"),
            "title": item.get("title"),
            "description": truncated_description, # Use truncated description
            "size": item.get("size"),
            "color": item.get("color"),
            "asin": item.get("asin")
        }
    )
    docs.append(doc)

# 🔹 Agregar documentos a la colección en Chroma Cloud en lotes más pequeños
batch_size = 300  # Ajustado para cumplir con la cuota de 300
for i in range(0, len(docs), batch_size):
    batch = docs[i:i + batch_size]
    vectorstore.add_documents(batch)
    print(f"✅ Se agregaron {len(batch)} documentos al lote {i//batch_size + 1}.")

print(f"✅ Se agregaron un total de {len(docs)} documentos a la base de datos Chroma Cloud.")

✅ Se agregaron 300 documentos al lote 1.
✅ Se agregaron 300 documentos al lote 2.
✅ Se agregaron 300 documentos al lote 3.
✅ Se agregaron 300 documentos al lote 4.
✅ Se agregaron 300 documentos al lote 5.
✅ Se agregaron 300 documentos al lote 6.
✅ Se agregaron 300 documentos al lote 7.
✅ Se agregaron 300 documentos al lote 8.
✅ Se agregaron 300 documentos al lote 9.
✅ Se agregaron 300 documentos al lote 10.
✅ Se agregaron 300 documentos al lote 11.
✅ Se agregaron 300 documentos al lote 12.
✅ Se agregaron 300 documentos al lote 13.
✅ Se agregaron 300 documentos al lote 14.
✅ Se agregaron 300 documentos al lote 15.
✅ Se agregaron 300 documentos al lote 16.
✅ Se agregaron 300 documentos al lote 17.
✅ Se agregaron 300 documentos al lote 18.
✅ Se agregaron 300 documentos al lote 19.
✅ Se agregaron 300 documentos al lote 20.
✅ Se agregaron 300 documentos al lote 21.
✅ Se agregaron 300 documentos al lote 22.
✅ Se agregaron 300 documentos al lote 23.
✅ Se agregaron 300 documentos al lote 24.
✅

KeyboardInterrupt: 

In [ ]:
docs[0:15]

[Document(metadata={'price': 'None', 'avg_rating': 4.8, 'title': 'Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)', 'description': '', 'size': '', 'color': '', 'asin': None}, page_content='Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)\n\n'),
 Document(metadata={'price': 'None', 'avg_rating': 4.5, 'title': 'Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.', 'description': '', 'size': '', 'color': '', 'asin': None}, page_content='Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.\n\n'),
 Document(metadata={'price': 'None', 'avg_rating': 4.4, 'title': 'Eye Patch Black Adult with Tie Band (6 Per Pack)', 'description': '', 'size': '', 'color': '', 'asin': None}, page_content='Eye Patch Black Adult with Tie Band (6 Per Pack)\n\n'),
 Document(metadata={'price': 'None', 'avg_rating': 3.1, 'title': '

In [ ]:
# 🔹 LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 🔹 Retriever (missing line!)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 🔹 Format function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 🔹 Prompt
prompt = ChatPromptTemplate.from_template("""
Recomienda en español basándote SOLO en el contexto. Rating alto + cualquier precio.

Contexto productos:
{context}

Pregunta: {input}

Recomendación clara con opciones:
""")

# 🔹 RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
# 🔹 Test
query = "Dime productos que valgan entre 20 y 22 euros"
result = rag_chain.invoke(query)
print("🎯 Recomendación:")
print(result)

# 🔹 Check sources
docs = retriever.invoke(query)
print("\n📄 Fuentes:")
for i, doc in enumerate(docs[:3], 1):
    print(f"{i}. {doc.page_content[:120]}...")

🎯 Recomendación:
Aquí tienes algunas recomendaciones de productos que podrían estar en el rango de 20 a 22 euros, basándome en el contexto que proporcionaste:

1. **Instituto Español Deodorant Avena & Leche Roll On Combo (6 Pack)**: Este combo de desodorantes es ideal para quienes buscan una protección efectiva y un aroma agradable. Perfecto tanto para hombres como para mujeres, y su formato en roll-on lo hace muy práctico.

2. **Avena New 328429 Bath & Shower Gel (25.5Z)**: Este gel de baño es una excelente opción para quienes desean una experiencia de ducha suave y nutritiva. Su fórmula con avena es ideal para pieles sensibles y proporciona una limpieza delicada.

Te recomiendo verificar los precios en tu tienda de confianza, ya que pueden variar. ¡Espero que encuentres lo que buscas!

📄 Fuentes:
1. Instituto Espanol Deodorant Avena & Leche Roll On Combo (6 Pack). 080585090197 antiperspirant roll-on beauty makeup men ...
2. avena New 328429 Bath & Shower Gel 25.5Z Instituto Espanol (

In [ ]:
query = "Recomiendame el mejor producto de belleza para bebes con piel sensible"
result = rag_chain.invoke(query)
print(" Recomendación:")
print(result)

# 🔹 Check sources
docs                      = retriever.invoke(query)
print("\n📄 Fuentes:")
for i, doc in enumerate(docs[:3], 1):
    print(f"{i}. {doc.page_content[:180]}...")

 Recomendación:
Para el cuidado de la piel sensible de tu bebé, te recomiendo el **Chef Kyle Milk Soothing Gel**. Este gel hidratante y suavizante es ideal para la piel delicada de los bebés, ya que está clínicamente aprobado y no contiene parabenos ni colorantes artificiales. Su fórmula suave y natural es perfecta para calmar la piel después de la exposición al sol o al viento frío, lo que lo convierte en una excelente opción para mantener la piel de tu bebé hidratada y protegida.

Otra opción que puedes considerar es la **Leche Limpiadora Hidratante de Natura Siberica**. Este producto está diseñado específicamente para pieles secas y sensibles, ofreciendo una limpieza suave y una hidratación profunda.

Ambos productos son de alta calidad y están diseñados para cuidar la piel sensible de los bebés, así que puedes elegir el que mejor se adapte a tus necesidades.

📄 Fuentes:
1. Para Mi Bebe Baby Cologne Tamano Familiar 25 oz - Imported From Spain (Pink-Girl)

Para Mi Bebe Colonia Infant

In [ ]:
from google.colab import drive
drive.mount('/content/mydrive/')

# Carpeta donde están tus datos

data_path = '/content/mydrive/MyDrive/ecommerce-agent'

Drive already mounted at /content/mydrive/; to attempt to forcibly remount, call drive.mount("/content/mydrive/", force_remount=True).


In [ ]:
import os
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# 🔹 CONFIG Chroma Cloud (TUS DATOS)
CHROMA_CONFIG = {
    CHROMA_API_KEY= userdata.get('CHROMA_API_KEY'),
    TENANT_KEY= userdata.get('TENANT_KEY'),
    'database': 'Amazon_database',
    'collection': 'langchain'
}



In [ ]:
# 🔹 Cliente + Vectorstore
client = chromadb.CloudClient(**{k: v for k, v in CHROMA_CONFIG.items() if k != 'collection'})
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    client=client,
    collection_name=CHROMA_CONFIG['collection'],
    embedding_function=embeddings
)



In [ ]:
def buscar_similar(query, k=5):
    """Busca productos similares"""
    docs = vectorstore.similarity_search(query, k=k)
    print(f"\n🔍 '{query}' - Top {k}:")
    for i, doc in enumerate(docs, 1):
        print(f"  {i}. {doc.page_content[:100]}...")
    return docs

def buscar_con_score(query, k=5):
    """Busca con scores de similitud"""
    docs_scores = vectorstore.similarity_search_with_score(query, k=k)
    print(f"\n📊 '{query}' - Top {k} (score):")
    for i, (doc, score) in enumerate(docs_scores, 1):
        print(f"  {i}. {doc.page_content[:80]}... (score: {score:.3f})")
    return docs_scores

# 🔹 USO
buscar_similar("desodorante")
buscar_con_score("crema hidratante", k=3)


🔍 'desodorante' - Top 5:
  1. Agree deodorant stick

...
  2. Linha Accordes Boticario - Desodorante Antitranspirante Aerosol 75 Gr - (Boticario Accordes Collecti...
  3. Deeoral - Desodorante NATURAL en Spray - Sin Aluminio - Elimina el mal olor corporal (1.01 Oz)

DESO...
  4. avena instituto espanol fresh deodorant roll on anti-transpirant

Avena instituto espanol Fresh frag...
  5. avena instituto espanol fresh deodorant roll on anti-transpirant

Avena instituto espanol Fresh frag...

📊 'crema hidratante' - Top 3 (score):
  1. Lactovit Mousse Creme, Crema Hidratante para la Cara y el Cuerpo Para Piel Norma... (score: 0.829)
  2. Crema Humectante de Manzana (3-Pack) 2.11oz / 60gr

... (score: 0.880)
  3. MARTIDERM CALAMINA PLUS CREMA 75 ML

... (score: 0.884)


[(Document(metadata={'description': '', 'avg_rating': 4.4, 'size': '', 'color': '', 'price': '13.57', 'title': 'Lactovit Mousse Creme, Crema Hidratante para la Cara y el Cuerpo Para Piel Normal a Seca, Nutritiva y Ligera, 8.5 oz (250ml)'}, page_content='Lactovit Mousse Creme, Crema Hidratante para la Cara y el Cuerpo Para Piel Normal a Seca, Nutritiva y Ligera, 8.5 oz (250ml)\n\n'),
  0.8291598),
 (Document(metadata={'avg_rating': 3.8, 'title': 'Crema Humectante de Manzana (3-Pack) 2.11oz / 60gr', 'price': '18.99', 'description': '', 'color': '', 'size': ''}, page_content='Crema Humectante de Manzana (3-Pack) 2.11oz / 60gr\n\n'),
  0.8797878),
 (Document(metadata={'size': '', 'price': '19.49', 'avg_rating': 4.4, 'color': '', 'title': 'MARTIDERM CALAMINA PLUS CREMA 75 ML', 'description': ''}, page_content='MARTIDERM CALAMINA PLUS CREMA 75 ML\n\n'),
  0.8836091)]

In [ ]:
import os
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
import chromadb

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 🔹 Chroma Cloud
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = chromadb.CloudClient(
    CHROMA_API_KEY= userdata.get('CHROMA_API_KEY'),
    TENANT_KEY= userdata.get('TENANT_KEY'),
    database='Amazon_database'
)
vectorstore = Chroma(
    client=client,
    collection_name="langchain",
    embedding_function=embeddings
)
print("✅ Chroma Cloud ready!")

# 🔹 LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 🔹 Retriever (missing line!)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 🔹 Format function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 🔹 Prompt
prompt = ChatPromptTemplate.from_template("""
Recomienda en español basándote SOLO en el contexto. Rating alto + cualquier precio.

Contexto productos:
{context}

Pregunta: {input}

Recomendación clara con opciones:
""")

# 🔹 RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def rag_amazon(query, filter_kwargs=None):
    """Busca productos en Amazon y devuelve recomendaciones basadas en la consulta del usuario.""" # Added docstring
    if filter_kwargs:
        retriever.search_kwargs["filter"] = filter_kwargs

    # Recuperar respuesta y docs
    respuesta = rag_chain.invoke(query)

    return respuesta

/tmp/ipython-input-31166/505310263.py:19: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


✅ Chroma Cloud ready!


In [ ]:
# PRUEBAS
rag_amazon("Recomiéndame un desodorante con buen rating y precio menor de 20 euros")
rag_amazon("Mejor crema hidratante")
rag_amazon("Productos para el pelo muy buenos")

'Si estás buscando productos para el cabello de alta calidad, aquí tienes algunas excelentes opciones:\n\n1. **Cirugía Capilat Kera Fruit Tratamiento Profesional (1 litro) + Shampoo Antiresiduo**: Este tratamiento es ideal si buscas alisar el cabello con resultados visibles. La combinación con el shampoo antiresiduo asegura que tu cabello esté limpio y preparado para recibir el tratamiento.\n\n2. **Queratina Gold Diamond, Cera Fría (1 Litro)**: Este producto es perfecto para quienes desean un alisado duradero y un cabello suave y brillante. La queratina ayuda a reparar el daño y a mantener el cabello saludable.\n\n3. **Savile Crema Para Peinar Liso Colagen (300ml)**: Si prefieres un producto para peinar que también ofrezca beneficios de alisado, esta crema con colágeno es una excelente opción. Ayuda a mantener el cabello liso y manejable.\n\n4. **Kaba KABA Shampoo de Cebolla + Bio mascarilla Capilar**: Este combo es ideal para quienes buscan fortalecer el cabello y promover su crecimie

In [ ]:
# 🔹 Definir la clase del carrito
class ShoppingCart:
    def __init__(self):
        # Lista de productos añadidos
        self.items = []

    def add_product(self, product):
        """
        product: dict con al menos 'title', 'price', 'asin' o ID único
        """
        self.items.append(product)
        print(f"✅ '{product['title']}' agregado al carrito.")

    def remove_product(self, product_id):
        """
        product_id: asin o id único del producto
        """
        for i, item in enumerate(self.items):
            if item.get('asin') == product_id:
                removed = self.items.pop(i)
                print(f"🗑 '{removed['title']}' eliminado del carrito.")
                return
        print(f"⚠ Producto con id {product_id} no encontrado en el carrito.")

    def view_cart(self):
        if not self.items:
            print("🛒 El carrito está vacío.")
            return
        print("🛒 Carrito de compras:")
        total = 0
        for idx, item in enumerate(self.items, 1):
            price = item.get("price", 0)
            total += price if isinstance(price, (int, float)) else 0
            print(f"{idx}. {item['title']} - Precio: {price} - Rating: {item.get('rating', 'N/A')}")
        print(f"💰 Total: {total}")

    def clear_cart(self):
        self.items = []
        print("🧹 Carrito vaciado.")


In [ ]:
import json

# Ruta del archivo JSON
json_path = "/content/mydrive/MyDrive/ecommerce-agent/data/processed/amazon_products_clean.json"

# Leer el JSON
with open(json_path, "r", encoding="utf-8") as f:
    products = [json.loads(line) for line in f]

# Filtrar productos que tengan precio válido (numérico o cadena convertible a número)
productos_con_precio = []
for p in products:
    price = p.get("price")
    try:
        price_value = float(price)
        productos_con_precio.append({
            "title": p.get("title", "Sin título"),
            "price": price_value,
            "rating": p.get("rating", "Sin rating"),
            "asin": p.get("asin", "Sin asin")
        })
    except (TypeError, ValueError):
        continue

# Mostrar los primeros 10 productos con precio
print(f"Productos con precio encontrados: {len(productos_con_precio)}\n")
for p in productos_con_precio[:10]:
    print(f"{p['title']} | Precio: {p['price']} | Rating: {p['rating']}")

Productos con precio encontrados: 17704

Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY | Precio: 6.99 | Rating: Sin rating
Gold extatic Musk EDT 90ml | Precio: 86.95 | Rating: Sin rating
Brand New Headrang Face line Contour V-line Machine | Precio: 79.5 | Rating: Sin rating
BioMiracle StarDust Pixie Bubble Mask, Clarifying Foaming Face Mask with Green Tea and Apple, Carbonated Bubble Cupro Sheet Mask for Clear, Even Skin | Precio: 5.99 | Rating: Sin rating
VIROCHEMISTRY Pheromones For Women (Elixir) - Elegant, Ultra Strength Organic Fragrance Body Perfume (1 Fl. Oz) | Precio: 29.8 | Rating: Sin rating
Garnier Fructis Color Sealer, Instant, Lightweight Leave-In, Color Shield, For Color-Treated Hair, 6 oz. | Precio: 24.0 | Rating: Sin rating
Enjoy VOLUMIZING ELIXIR, Style (with Sleek Steel Pin Tail Comb) (8.8 oz) | Precio: 22.49 | Rating: Sin rating
Suavecito X Breast Cancer Solutions - Original Hold Pomade | Precio: 11.99 | Rating: Sin rating


In [ ]:
cart = ShoppingCart()

# Lista de productos que mencionaste
productos = [
    {"title": "Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY", "price": 6.99, "rating": None},
    {"title": "Gold extatic Musk EDT 90ml", "price": 86.95, "rating": None},
    {"title": "Brand New Headrang Face line Contour V-line Machine", "price": 79.5, "rating": None},
    {"title": "BioMiracle StarDust Pixie Bubble Mask, Clarifying Foaming Face Mask with Green Tea and Apple, Carbonated Bubble Cupro Sheet Mask for Clear, Even Skin", "price": 5.99, "rating": None},
    {"title": "VIROCHEMISTRY Pheromones For Women (Elixir) - Elegant, Ultra Strength Organic Fragrance Body Perfume (1 Fl. Oz)", "price": 29.8, "rating": None},
    {"title": "Garnier Fructis Color Sealer, Instant, Lightweight Leave-In, Color Shield, For Color-Treated Hair, 6 oz.", "price": 24.0, "rating": None},
    {"title": "Enjoy VOLUMIZING ELIXIR, Style (with Sleek Steel Pin Tail Comb) (8.8 oz)", "price": 22.49, "rating": None}
]

# Añadir productos al carrito
for p in productos:
    cart.add_product(p)

# Ver contenido del carrito
cart.view_cart()

✅ 'Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY' agregado al carrito.
✅ 'Gold extatic Musk EDT 90ml' agregado al carrito.
✅ 'Brand New Headrang Face line Contour V-line Machine' agregado al carrito.
✅ 'BioMiracle StarDust Pixie Bubble Mask, Clarifying Foaming Face Mask with Green Tea and Apple, Carbonated Bubble Cupro Sheet Mask for Clear, Even Skin' agregado al carrito.
✅ 'VIROCHEMISTRY Pheromones For Women (Elixir) - Elegant, Ultra Strength Organic Fragrance Body Perfume (1 Fl. Oz)' agregado al carrito.
✅ 'Garnier Fructis Color Sealer, Instant, Lightweight Leave-In, Color Shield, For Color-Treated Hair, 6 oz.' agregado al carrito.
✅ 'Enjoy VOLUMIZING ELIXIR, Style (with Sleek Steel Pin Tail Comb) (8.8 oz)' agregado al carrito.
🛒 Carrito de compras:
1. Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY - Precio: 6.99 - Rating: None
2. Gold extatic Musk EDT 90ml - Precio: 86.95 - Rating: None
3. Brand N

In [ ]:
import re
import json
import os
from typing import Dict, Any, Tuple, Optional

class ShippingDataTool:
    """
    Herramienta para recopilar y validar datos del cliente y dirección de envío.
    Soporta:
      - Recepción de datos como dict y validación.
      - Modo interactivo por consola para pedir datos al usuario.
      - Guardado/lectura de los datos en/desde un archivo JSON.
    """

    EMAIL_RE = re.compile(r"^[^@ \t\r\n]+@[^@ \t\r\n]+\.[^@ \t\r\n]+$")
    PHONE_RE = re.compile(r"^[0-9+\-\s()]{6,20}$")  # aceptable para la mayoría de teléfonos
    POSTAL_RE = re.compile(r"^[A-Za-z0-9\s\-]{3,10}$")  # genérico, admite códigos alfanuméricos

    def __init__(self, storage_file: str = "shipping_data.json"):
        self.storage_file = storage_file

    def validate(self, data: Dict[str, Any]) -> Tuple[bool, Dict[str, str]]:
        """
        Valida un diccionario con los campos esperados.
        Devuelve (es_valido, errores) donde errores es un dict campo->mensaje.
        Campos esperados:
          - nombre (required)
          - email (required)
          - telefono (optional pero recomendado)
          - direccion_line1 (required)
          - direccion_line2 (optional)
          - ciudad (required)
          - estado_provincia (optional)
          - codigo_postal (required)
          - pais (required)
          - metodo_envio (optional)
          - instrucciones (optional)
        """
        errores = {}

        nombre = (data.get("nombre") or "").strip()
        if not nombre:
            errores["nombre"] = "El nombre es obligatorio."

        email = (data.get("email") or "").strip()
        if not email or not self.EMAIL_RE.match(email):
            errores["email"] = "Email inválido o vacío."

        telefono = (data.get("telefono") or "").strip()
        if telefono and not self.PHONE_RE.match(telefono):
            errores["telefono"] = "Teléfono con formato no válido."

        direccion_line1 = (data.get("direccion_line1") or "").strip()
        if not direccion_line1:
            errores["direccion_line1"] = "La dirección (línea 1) es obligatoria."

        ciudad = (data.get("ciudad") or "").strip()
        if not ciudad:
            errores["ciudad"] = "La ciudad es obligatoria."

        codigo_postal = (data.get("codigo_postal") or "").strip()
        if not codigo_postal or not self.POSTAL_RE.match(codigo_postal):
            errores["codigo_postal"] = "Código postal inválido o vacío."

        pais = (data.get("pais") or "").strip()
        if not pais:
            errores["pais"] = "El país es obligatorio."

        # Retornar
        return (len(errores) == 0, errores)

    def from_input(self) -> Dict[str, Any]:
        """
        Modo interactivo por consola para pedir los datos al usuario.
        Devuelve un dict con los datos (no hace reintentos automáticos).
        """
        print("Por favor, introduce los datos de envío y del cliente.")
        nombre = input("Nombre completo: ").strip()
        email = input("Email: ").strip()
        telefono = input("Teléfono (opcional): ").strip()
        direccion_line1 = input("Dirección - línea 1: ").strip()
        direccion_line2 = input("Dirección - línea 2 (opcional): ").strip()
        ciudad = input("Ciudad: ").strip()
        estado_provincia = input("Estado/Provincia (opcional): ").strip()
        codigo_postal = input("Código postal: ").strip()
        pais = input("País: ").strip()
        metodo_envio = input("Método de envío (estándar/express) (opcional): ").strip()
        instrucciones = input("Instrucciones de entrega (opcional): ").strip()

        data = {
            "nombre": nombre,
            "email": email,
            "telefono": telefono or None,
            "direccion_line1": direccion_line1,
            "direccion_line2": direccion_line2 or None,
            "ciudad": ciudad,
            "estado_provincia": estado_provincia or None,
            "codigo_postal": codigo_postal,
            "pais": pais,
            "metodo_envio": metodo_envio or None,
            "instrucciones": instrucciones or None
        }
        return data

    def collect_and_validate(self, data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Valida los datos recibidos; si son válidos los normaliza y devuelve el dict limpio.
        Si hay errores, lanza ValueError con el detalle de errores.
        """
        valido, errores = self.validate(data)
        if not valido:
            raise ValueError(json.dumps(errores, ensure_ascii=False))
        # Normalizar algunos campos (ejemplo)
        clean = {
            "nombre": data.get("nombre").strip(),
            "email": data.get("email").strip(),
            "telefono": (data.get("telefono") or "").strip() or None,
            "direccion_line1": data.get("direccion_line1").strip(),
            "direccion_line2": (data.get("direccion_line2") or "").strip() or None,
            "ciudad": data.get("ciudad").strip(),
            "estado_provincia": (data.get("estado_provincia") or "").strip() or None,
            "codigo_postal": data.get("codigo_postal").strip(),
            "pais": data.get("pais").strip(),
            "metodo_envio": (data.get("metodo_envio") or "estándar").strip(),
            "instrucciones": (data.get("instrucciones") or "").strip() or None
        }
        return clean

    def save(self, data: Dict[str, Any]) -> None:
        """
        Guarda los datos validados en self.storage_file (sobrescribe).
        """
        dirpath = os.path.dirname(os.path.abspath(self.storage_file))
        if dirpath and not os.path.exists(dirpath):
            os.makedirs(dirpath, exist_ok=True)
        with open(self.storage_file, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    def load(self) -> Optional[Dict[str, Any]]:
        """
        Lee y devuelve los datos guardados, o None si no existe.
        """
        if not os.path.exists(self.storage_file):
            return None
        with open(self.storage_file, "r", encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                return None


In [ ]:
# Crear la instancia
tool = ShippingDataTool(storage_file="/content/shipping_data_test.json")

# Datos simulados (puedes cambiarlos)
datos_prueba = {
    "nombre": "Carlos López",
    "email": "carlos.lopez@example.com",
    "telefono": "+34 611 222 333",
    "direccion_line1": "Avenida de los Pinos 45",
    "direccion_line2": "Piso 3, Puerta B",
    "ciudad": "Valencia",
    "estado_provincia": "Valencia",
    "codigo_postal": "46021",
    "pais": "España",
    "metodo_envio": "express",
    "instrucciones": "Entregar por la tarde, si es posible."
}

# Validar y guardar los datos
try:
    datos_validados = tool.collect_and_validate(datos_prueba)
    print("✅ Datos validados correctamente:")
    print(datos_validados)
    tool.save(datos_validados)
except ValueError as e:
    print("❌ Error de validación:", e)

# Cargar los datos guardados
print("\n📦 Datos cargados desde el archivo:")
print(tool.load())


✅ Datos validados correctamente:
{'nombre': 'Carlos López', 'email': 'carlos.lopez@example.com', 'telefono': '+34 611 222 333', 'direccion_line1': 'Avenida de los Pinos 45', 'direccion_line2': 'Piso 3, Puerta B', 'ciudad': 'Valencia', 'estado_provincia': 'Valencia', 'codigo_postal': '46021', 'pais': 'España', 'metodo_envio': 'express', 'instrucciones': 'Entregar por la tarde, si es posible.'}

📦 Datos cargados desde el archivo:
{'nombre': 'Carlos López', 'email': 'carlos.lopez@example.com', 'telefono': '+34 611 222 333', 'direccion_line1': 'Avenida de los Pinos 45', 'direccion_line2': 'Piso 3, Puerta B', 'ciudad': 'Valencia', 'estado_provincia': 'Valencia', 'codigo_postal': '46021', 'pais': 'España', 'metodo_envio': 'express', 'instrucciones': 'Entregar por la tarde, si es posible.'}


In [ ]:
import json
from typing import Dict, Any
# from shipping_data_tool import ShippingDataTool   # tu clase de dirección
# from herramienta_pago import HerramientaPago, ShoppingCart  # tus clases de pago/carrito

# Asumiendo que ShoppingCart ya está definido en el entorno
# Y definiendo una clase placeholder para HerramientaPago

class HerramientaPago:
    def __init__(self, carrito, archivo_ordenes: str = "orders.json"):
        self.carrito = carrito
        self.archivo_ordenes = archivo_ordenes
        self._ensure_orders_file_exists()

    def _ensure_orders_file_exists(self):
        if not os.path.exists(self.archivo_ordenes):
            with open(self.archivo_ordenes, 'w', encoding='utf-8') as f:
                json.dump([], f) # Initialize with an empty list

    def pagar_con_tarjeta(self, tarjeta: Dict, datos_cliente: Dict) -> Dict:
        if not self.carrito.items:
            return {"status": "fallido", "mensaje": "El carrito está vacío."}

        # Simular proceso de pago
        total = sum(item.get("price", 0) for item in self.carrito.items)
        if total == 0:
             return {"status": "fallido", "mensaje": "No se pueden pagar productos sin precio."}

        # Generar un ID de orden simple
        order_id = f"ORD-{len(self.obtener_ordenes()) + 1}-{int(time.time())}"

        order = {
            "orden_id": order_id,
            "cliente": datos_cliente,
            "productos": self.carrito.items,
            "total": round(total, 2),
            "moneda": "EUR", # Asumiendo euros por los precios de ejemplo
            "metodo_pago": "tarjeta",
            "tarjeta_info": {"last4": tarjeta["number"][-4:], "exp_month": tarjeta["exp_month"], "exp_year": tarjeta["exp_year"]},
            "status": "procesando",
            "timestamp": datetime.now().isoformat()
        }

        self._guardar_orden(order)
        self.carrito.clear_cart()
        return {"status": "exitoso", "mensaje": f"Orden {order_id} creada y pago procesado.", "orden": order}

    def obtener_ordenes(self) -> list:
        with open(self.archivo_ordenes, 'r', encoding='utf-8') as f:
            return json.load(f)

    def _guardar_orden(self, nueva_orden: Dict) -> None:
        ordenes = self.obtener_ordenes()
        ordenes.append(nueva_orden)
        with open(self.archivo_ordenes, 'w', encoding='utf-8') as f:
            json.dump(ordenes, f, ensure_ascii=False, indent=2)


import os
import time
from datetime import datetime


# ======================================================
# 1️⃣  Crear las instancias de las tools
# ======================================================
carrito = ShoppingCart()
tool_envio = ShippingDataTool(storage_file="/content/shipping_data_cliente.json")
herramienta_pago = HerramientaPago(carrito, archivo_ordenes="/content/orders.json")

# ======================================================
# 2️⃣  Añadir productos al carrito
# ======================================================
productos = [
    {"title": "Lurrose 100Pcs Full Cover Fake Toenails", "price": 6.99},
    {"title": "Gold extatic Musk EDT 90ml", "price": 86.95}
]

for p in productos:
    carrito.add_product(p)

print("Carrito actual:")
# Modificado: view_cart ya imprime, no es necesario iterar y reimprimir
carrito.view_cart()

# ======================================================
# 3️⃣  Capturar y validar datos de envío
# ======================================================
datos_cliente = {
    "nombre": "Lucía Fernández",
    "email": "lucia.fernandez@example.com",
    "telefono": "+34 611 222 333",
    "direccion_line1": "Calle del Río 12",
    "direccion_line2": "3ºA",
    "ciudad": "Madrid",
    "estado_provincia": "Comunidad de Madrid",
    "codigo_postal": "28045",
    "pais": "España",
    "metodo_envio": "express",
    "instrucciones": "Llamar antes de entregar."
}

try:
    datos_validados = tool_envio.collect_and_validate(datos_cliente)
    tool_envio.save(datos_validados)
    print("\nDatos de envío validados y guardados correctamente.")
except ValueError as e:
    print("Error en los datos de envío:", e)
    raise SystemExit()

# ======================================================
# 4️⃣  Simular pago (tarjeta o PayPal)
# ======================================================
tarjeta = {
    "number": "4242424242424242",  # número de prueba válido (Luhn)
    "exp_month": 12,
    "exp_year": 2030,
    "cvc": "123",
    "name": datos_validados["nombre"]
}

# 💳 Pagar con tarjeta
resultado_pago = herramienta_pago.pagar_con_tarjeta(tarjeta, datos_validados)

# ======================================================
# 5️⃣  Mostrar resultado
# ======================================================
print("\nResultado del pago y creación de orden:\n")
print(json.dumps(resultado_pago, indent=2, ensure_ascii=False))

# ======================================================
# 6️⃣  Ver todas las órdenes guardadas
# ======================================================
print("\nÓrdenes almacenadas actualmente:")
for orden in herramienta_pago.obtener_ordenes():
    print(f"- {orden['orden_id']} | Cliente: {orden['cliente']['nombre']} | Total: {orden['total']} {orden['moneda']}")

✅ 'Lurrose 100Pcs Full Cover Fake Toenails' agregado al carrito.
✅ 'Gold extatic Musk EDT 90ml' agregado al carrito.
Carrito actual:
🛒 Carrito de compras:
1. Lurrose 100Pcs Full Cover Fake Toenails - Precio: 6.99 - Rating: N/A
2. Gold extatic Musk EDT 90ml - Precio: 86.95 - Rating: N/A
💰 Total: 93.94

Datos de envío validados y guardados correctamente.
🧹 Carrito vaciado.

Resultado del pago y creación de orden:

{
  "status": "exitoso",
  "mensaje": "Orden ORD-1-1772271304 creada y pago procesado.",
  "orden": {
    "orden_id": "ORD-1-1772271304",
    "cliente": {
      "nombre": "Lucía Fernández",
      "email": "lucia.fernandez@example.com",
      "telefono": "+34 611 222 333",
      "direccion_line1": "Calle del Río 12",
      "direccion_line2": "3ºA",
      "ciudad": "Madrid",
      "estado_provincia": "Comunidad de Madrid",
      "codigo_postal": "28045",
      "pais": "España",
      "metodo_envio": "express",
      "instrucciones": "Llamar antes de entregar."
    },
    "product

In [ ]:
# ----------------------------
# 1️⃣ Imports
# ----------------------------
from langchain_core.tools import tool, Tool # Added Tool for explicit wrapping
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# ----------------------------
# 2️⃣ Carrito y datos de envío
# ----------------------------
carrito = {}
direccion_envio = {}

# ----------------------------
# 3️⃣ Tools
# ----------------------------

@tool
def ver_carrito() -> str:
    """Muestra los productos actualmente en el carrito de compras."""
    if (not carrito) or  (len(carrito)==0):
        return "El carrito está vacío."
    return "\n".join([f"{i+1}. {item['title']} - Precio: {item['price']}"
                      for i, item in enumerate(carrito.values())])

@tool
def agregar_producto(title: str, price: float) -> str:
    """Agrega un producto con un título y un precio especificados al carrito de compras."""
    carrito[title] = {"title": title, "price": price}
    return f"Producto '{title}' agregado al carrito."

@tool
def guardar_direccion(nombre: str, email: str, direccion: str) -> str:
    """Guarda la información de la dirección de envío del cliente."""
    direccion_envio.update({"nombre": nombre, "email": email, "direccion": direccion})
    return f"Dirección guardada para {nombre}."

@tool
def pagar_tarjeta(nombre: str, numero: str, mes: int, año: int, cvc: str) -> str:
    """Procesa un pago con tarjeta de crédito para los productos en el carrito."""
    if not carrito:
        return "No hay productos en el carrito para pagar."
    total = sum([float(item['price'].replace('$','')) for item in carrito.values()])
    carrito.clear()
    return f"Pago realizado con tarjeta {numero[-4:]} por un total de ${total}. ¡Gracias {nombre}!"

# Tool de búsqueda de productos con sus características

# List of all tools
tools = [
    ver_carrito,
    agregar_producto,
   # guardar_direccion,
   # pagar_tarjeta,
    Tool.from_function(
        func=rag_amazon,
        name="rag_amazon",
        description="Busca productos en Amazon y devuelve recomendaciones basadas en la consulta del usuario."
    ) # Explicitly wrap rag_amazon as a Tool
]

# ----------------------------
# 4️⃣ Crear el agente
# ----------------------------
chat_model = ChatOpenAI(model="gpt-5.1", temperature=0)

agent = create_agent(
    chat_model,
    tools, # Use the defined tools list
    system_prompt="Eres un asistente de compras. Usa las funciones correctamente según lo que el usuario pida."
)

# ----------------------------
# 5️⃣ Función de chat interactivo
# ----------------------------

messages=[]
def chat_ecommerce():
    print("¡Bienvenido al asistente de compras! Escribe 'salir' para terminar.\n")
    while True:
        entrada = input("Tú: ")
        messages.append({"role": "user", "content": entrada})
        result=agent.invoke({"message":messages})
        messages.append(result['messages'][-1])
        print("chatbot: " + result["messages"][-1].content)
        if entrada.lower() in ["salir", "exit"]:
            print("Agente: Gracias por usar el asistente de compras. ¡Hasta luego!")

In [ ]:
def validar_tools(agent, tools):
    print("\n🧠 Validando tools registradas...")
    print(f"Total declaradas: {len(tools)}")
    for t in tools:
        if hasattr(t, 'name'):
            print(f" - {t.name}")
        else:
            print(f" - {str(t)} (Advertencia: No es un objeto Tool de LangChain o le falta el atributo 'name')")
    print("✅ Estas herramientas fueron asociadas al agente.")

# Ejecutar validación
validar_tools(agent, tools)


🧠 Validando tools registradas...
Total declaradas: 6
 - ver_carrito
 - agregar_producto
 - guardar_direccion
 - pagar_tarjeta
 - buscar_producto
 - <function rag_amazon at 0x7d66ab9977e0> (Advertencia: No es un objeto Tool de LangChain o le falta el atributo 'name')
✅ Estas herramientas fueron asociadas al agente.


In [ ]:
print(agent.nodes)

{'__start__': <langgraph.pregel._read.PregelNode object at 0x7d66b5127b00>, 'model': <langgraph.pregel._read.PregelNode object at 0x7d66b5127a10>, 'tools': <langgraph.pregel._read.PregelNode object at 0x7d66b359a810>}


In [ ]:
#messages.append({"role": "user", "content": "un jabon de glicerina por menos de 10 euros"})
result = agent.invoke({"message": [{"role": "user", "content": "buscame un jabon de glicerina por menos de 10 euros"}]})
#result["messages"][-1].content

In [ ]:
result

{'messages': [AIMessage(content='Puedo ayudarte a buscar productos, compararlos y llenar tu carrito.  \nCuéntame:\n\n- ¿Qué quieres comprar? (ej. “portátil para trabajo y juegos”, “tenis para correr”, “audífonos inalámbricos”)\n- Presupuesto aproximado\n- País donde compras (para afinar mejor los resultados)\n- Algún requisito clave (marca, tamaño, color, características, etc.)\n\nCon eso buscaré opciones en Amazon, te recomendaré modelos concretos y, si quieres, iré agregando productos al carrito.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 215, 'total_tokens': 336, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DECgYaQr3xsB0xv9dTzUwWNzg6iZN', 'servic

In [ ]:
import pandas as pd
import os
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import Tool, tool # Import Tool for explicit wrapping

# Tus herramientas existentes (ver_carrito, agregar_producto, guardar_direccion, pagar_tarjeta, buscar_producto)
# deben estar definidas en celdas anteriores y ser @tool decoradas o explícitamente Tool.from_function

# Asumiendo que rag_amazon también está definida y decorada con @tool o es una Tool.from_function

# Agrupamos las tools existentes y las nuevas.
# Asegúrate de que las funciones a continuación estén definidas en las celdas anteriores
# Si 'buscar_similar' o 'buscar_con_score' no están decoradas con @tool en sus definiciones,
# deben ser envueltas explícitamente como se hace con 'rag_amazon'.

tools_carrito = [ver_carrito, agregar_producto]
tools_pago_envio = [guardar_direccion, pagar_tarjeta]

# Asegúrate de que buscar_similar y buscar_con_score sean Tools
tools_busqueda = [
    buscar_producto,
    Tool.from_function(
        func=buscar_similar,
        name="buscar_similar",
        description="Busca productos similares a una query y devuelve los resultados."
    ),
    Tool.from_function(
        func=buscar_con_score,
        name="buscar_con_score",
        description="Busca productos similares a una query y devuelve los resultados con su score de similitud."
    ),
    Tool.from_function(
        func=rag_amazon,
        name="rag_amazon",
        description="Busca productos en Amazon y devuelve recomendaciones basadas en la consulta del usuario."
    )
]

# ✅ Reunimos todas las tools en una sola lista
tools = tools_carrito + tools_pago_envio + tools_busqueda

# Crear modelo de chat
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Crear agente
agent = create_agent(
    chat_model,
    tools,
    system_prompt=(
        "Eres un asistente de compras inteligente. "
        "Ayudas a los clientes a buscar productos, comparar, agregar al carrito, "
        "guardar dirección y procesar pagos. Usa las herramientas de forma adecuada según la consulta."
    )
)

In [ ]:
print("✅ Tools cargadas en el agente:")
for t in tools:
    print(f" - {t.name}")


✅ Tools cargadas en el agente:
 - ver_carrito
 - agregar_producto
 - guardar_direccion
 - pagar_tarjeta
 - buscar_producto
 - buscar_similar
 - buscar_con_score
 - rag_amazon


In [ ]:
chat_ecommerce()

¡Bienvenido al asistente de compras! Escribe 'salir' para terminar.

Tú: Hola
chatbot: ¿En qué puedo ayudarte con tus compras hoy? Puedo:

- Buscar productos en Amazon y recomendarte opciones.
- Agregar productos a tu carrito (con título y precio).
- Mostrarte lo que ya tienes en el carrito.

Dime qué estás buscando (por ejemplo: “audífonos inalámbricos hasta 50 euros para hacer deporte”) o qué acción quieres hacer con el carrito.
Tú: Quiero un gel para pieles sensibles 
chatbot: Puedo ayudarte a buscar y elegir productos, comparar opciones y armar tu carrito.  

Dime:
- Qué estás buscando (por ejemplo: “audífonos bluetooth con cancelación de ruido”, “laptop para universidad”, “regalo para niño de 8 años con presupuesto de 30€”).
- Tu presupuesto aproximado.
- Tus preferencias clave (marca, tamaño, color, características especiales, etc.).

Con eso puedo:
- Buscar opciones en Amazon.
- Sugerirte productos concretos.
- Añadir los que te gusten al carrito o mostrar lo que ya tienes en el

KeyboardInterrupt: Interrupted by user